In [1]:
import joblib

X_train = joblib.load(
    "X_train_random.pkl"
)

X_test = joblib.load(
    "X_test_random.pkl"
)

y_train = joblib.load(
    "y_train_random.pkl"
)

y_test = joblib.load(
    "y_test_random.pkl"
)

In [2]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_regression
)

In [3]:
preprocessor = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=f_regression,
            k=1000
        )
    )

])

In [8]:
import optuna

from lightgbm import LGBMRegressor

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

In [ ]:

import numpy as np

def objective(trial):

    model = LGBMRegressor(

        n_estimators=trial.suggest_int(
            "n_estimators",
            200,
            1500
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        num_leaves=trial.suggest_int(
            "num_leaves",
            20,
            300
        ),

        min_child_samples=trial.suggest_int(
            "min_child_samples",
            5,
            100
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            0,
            10
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            0,
            10
        ),

        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    pipeline = Pipeline([

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            model
        )

    ])

    scores = cross_validate(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring={

            "r2": "r2",

            "mae": "neg_mean_absolute_error"

        },

        n_jobs=-1

    )

    mean_r2 = np.mean(
        scores["test_r2"]
    )

    mean_mae = -np.mean(
        scores["test_mae"]
    )

    trial.set_user_attr(
        "mean_r2",
        mean_r2
    )

    trial.set_user_attr(
        "mean_mae",
        mean_mae
    )

    print(
        f"Trial {trial.number} | "
        f"Mean R2 = {mean_r2:.4f} | "
        f"Mean MAE = {mean_mae:.4f}"
    )

    return mean_r2

In [10]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-06-08 19:24:39,405] A new study created in memory with name: no-name-12c96fcb-0a3e-42b8-93fe-3efc6f6e0759


  0%|          | 0/30 [00:00<?, ?it/s]

Trial 0 | Mean R2 = 0.7972 | Mean MAE = 0.7064
[I 2026-06-08 19:30:17,711] Trial 0 finished with value: 0.7971722363927045 and parameters: {'n_estimators': 861, 'learning_rate': 0.012387255900948713, 'max_depth': 13, 'num_leaves': 98, 'min_child_samples': 53, 'subsample': 0.6773226548380518, 'colsample_bytree': 0.7712581527264397, 'reg_alpha': 7.366717054023417, 'reg_lambda': 0.01629711493132957}. Best is trial 0 with value: 0.7971722363927045.
Trial 1 | Mean R2 = 0.7928 | Mean MAE = 0.7067
[I 2026-06-08 19:33:43,530] Trial 1 finished with value: 0.7927948511685508 and parameters: {'n_estimators': 852, 'learning_rate': 0.08685614153294546, 'max_depth': 14, 'num_leaves': 107, 'min_child_samples': 86, 'subsample': 0.7753617203216898, 'colsample_bytree': 0.7590607294286353, 'reg_alpha': 5.660905332121269, 'reg_lambda': 1.3923068945213124}. Best is trial 0 with value: 0.7971722363927045.
Trial 2 | Mean R2 = 0.7785 | Mean MAE = 0.7451
[I 2026-06-08 19:34:41,858] Trial 2 finished with value:

In [11]:
print("Best Mean CV R2:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best Mean CV R2:
0.8000341298884119

Best Parameters:
{'n_estimators': 1402, 'learning_rate': 0.015213162868776496, 'max_depth': 7, 'num_leaves': 238, 'min_child_samples': 61, 'subsample': 0.9578486268536361, 'colsample_bytree': 0.6422902514180014, 'reg_alpha': 5.540119448046517, 'reg_lambda': 1.0514787338804057}


In [12]:
best_lgbm_pipeline = Pipeline([

    (
        "preprocessing",
        preprocessor
    ),

    (
        "model",

        LGBMRegressor(

            **study.best_params,

            random_state=42,

            n_jobs=-1,

            verbose=-1
        )
    )

])

In [13]:
best_lgbm_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('imputer', ...), ('variance_filter', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None


In [14]:
y_train_pred = best_lgbm_pipeline.predict(
    X_train
)

y_test_pred = best_lgbm_pipeline.predict(
    X_test
)

c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [18]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [19]:
train_r2 = r2_score(
    y_train,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

In [20]:
print("\n TUNED LIGHTGBM ")

print("\nTrain R2 :", train_r2)
print("Train RMSE :", train_rmse)
print("Train MAE :", train_mae)

print("\nTest R2 :", test_r2)
print("Test RMSE :", test_rmse)
print("Test MAE :", test_mae)


 TUNED LIGHTGBM 

Train R2 : 0.9275909737700555
Train RMSE : 0.6398055508604502
Train MAE : 0.386765577781781

Test R2 : 0.8263960766749234
Test RMSE : 0.9701588015433897
Test MAE : 0.6379423633367385


In [21]:
import pandas as pd

trials_df = study.trials_dataframe()

trials_df["Mean_CV_MAE"] = [
    t.user_attrs.get("mean_mae")
    for t in study.trials
]

trials_df["Mean_CV_R2"] = [
    t.user_attrs.get("mean_r2")
    for t in study.trials
]

final_trials_df = trials_df[
    [
        "number",
        "Mean_CV_R2",
        "Mean_CV_MAE",
        "params_n_estimators",
        "params_learning_rate",
        "params_max_depth",
        "params_num_leaves",
        "params_min_child_samples",
        "params_subsample",
        "params_colsample_bytree",
        "params_reg_alpha",
        "params_reg_lambda",
        "state"
    ]
].copy()

final_trials_df.columns = [
    "Trial_Number",
    "Mean_CV_R2",
    "Mean_CV_MAE",
    "N_Estimators",
    "Learning_Rate",
    "Max_Depth",
    "Num_Leaves",
    "Min_Child_Samples",
    "Subsample",
    "Colsample_Bytree",
    "Reg_Alpha",
    "Reg_Lambda",
    "Status"
]

final_trials_df.to_csv(
    "lgbm_optuna_trials.csv",
    index=False
)

final_trials_df.head()

,Trial_Number,Mean_CV_R2,Mean_CV_MAE,N_Estimators,Learning_Rate,Max_Depth,Num_Leaves,Min_Child_Samples,Subsample,Colsample_Bytree,Reg_Alpha,Reg_Lambda,Status
0,0,0.797172,0.706363,861,0.012387,13,98,53,0.677323,0.771258,7.366717,0.016297,COMPLETE
1,1,0.792795,0.706652,852,0.086856,14,107,86,0.775362,0.759061,5.660905,1.392307,COMPLETE
2,2,0.778497,0.745137,1070,0.279300,15,54,30,0.879372,0.799219,6.979376,4.640395,COMPLETE
3,3,0.798673,0.708193,939,0.014437,10,77,94,0.626649,0.726196,8.977661,4.456845,COMPLETE
4,4,0.796706,0.713469,1040,0.029428,5,109,54,0.881364,0.819095,2.127821,7.439481,COMPLETE


In [22]:
import joblib

joblib.dump(
    best_lgbm_pipeline,
    "best_lgbm_pipeline.pkl"
)

['best_lgbm_pipeline.pkl']

In [23]:
metrics_df = pd.DataFrame({

    "Metric": ["R2", "MAE", "RMSE"],

    "Train": [
        train_r2,
        train_mae,
        train_rmse
    ],

    "Test": [
        test_r2,
        test_mae,
        test_rmse
    ]

})

metrics_df.to_csv(
    "lgbm_final_metrics.csv",
    index=False
)

In [24]:
import pandas as pd

lgbm = pd.read_csv("lgbm_optuna_trials.csv")

lgbm

,Trial_Number,Mean_CV_R2,Mean_CV_MAE,N_Estimators,Learning_Rate,Max_Depth,Num_Leaves,Min_Child_Samples,Subsample,Colsample_Bytree,Reg_Alpha,Reg_Lambda,Status
0,0,0.797172,0.706363,861,0.012387,13,98,53,0.677323,0.771258,7.366717,0.016297,COMPLETE
1,1,0.792795,0.706652,852,0.086856,14,107,86,0.775362,0.759061,5.660905,1.392307,COMPLETE
2,2,0.778497,0.745137,1070,0.279300,15,54,30,0.879372,0.799219,6.979376,4.640395,COMPLETE
3,3,0.798673,0.708193,939,0.014437,10,77,94,0.626649,0.726196,8.977661,4.456845,COMPLETE
4,4,0.796706,0.713469,1040,0.029428,5,109,54,0.881364,0.819095,2.127821,7.439481,COMPLETE
5,5,0.792726,0.702978,1314,0.048970,15,264,68,0.742279,0.673817,2.882179,6.594755,COMPLETE
6,6,0.791402,0.713758,1041,0.100438,5,137,85,0.940015,0.865633,4.202494,0.329385,COMPLETE
7,7,0.784284,0.749640,679,0.020689,4,204,81,0.689241,0.850399,4.499549,6.149329,COMPLETE
8,8,0.797316,0.703107,831,0.048384,14,105,77,0.862528,0.734627,9.992990,7.127219,COMPLETE
9,9,0.784830,0.750875,761,0.010908,5,43,54,0.707265,0.811395,4.752622,8.311594,COMPLETE


In [25]:
lgbm.loc[
    lgbm["Mean_CV_R2"].idxmax()
]

Trial_Number               23
Mean_CV_R2           0.800034
Mean_CV_MAE           0.70004
N_Estimators             1402
Learning_Rate        0.015213
Max_Depth                   7
Num_Leaves                238
Min_Child_Samples          61
Subsample            0.957849
Colsample_Bytree      0.64229
Reg_Alpha            5.540119
Reg_Lambda           1.051479
Status               COMPLETE
Name: 23, dtype: object